# Chuvash (chv) — Tokenisation and Morphological Analysis

Chuvash (Oghur branch) is supported via Cyrillic script tokenisation and Prototype-quality Apertium FST morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation. Chuvash belongs to the Oghur branch and is distinct from all other Turkic languages.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('chv')

## 2. Tokenisation

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Chuvash Cyrillic text
cyrl = "Эпĕ шкулта вĕренетĕп."
print("Script Detection:")
print(f"  Detected: {detect_script(cyrl).name}")
print()

# Cyrillic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("chv", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(cyrl)
    print(f"Cyrillic:        {cyrl}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Cyrillic
    t_back = Transliterator("chv", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    cyrl_restored = t_back.transliterate(common)
    print(f"Restored:        {cyrl_restored}")
    print(f"Round-trip match: {cyrl == cyrl_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Chuvash: {e}")
    print("  For Cyrillic-based languages, use Script.LATIN as alternative")

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

Chuvash uses Cyrillic script with special characters (ă, ĕ, ç, ş). The transliteration system enables conversion to Latin.

In [ ]:
nlp_tok = Pipeline("chv", processors=["tokenize"])
doc = nlp_tok("Эпĕ шкулта вĕренетĕп.")
print([w.text for w in doc.words])

## 3. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "chv",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Эпĕ шкулта вĕренетĕп.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. Translation via NLLB-200

In [ ]:
turkicnlp.download("chv", processors=["translate"])
trans = Pipeline("chv", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Эпĕ шкулта вĕренетĕп.")
print("EN:", doc.translation)